In [0]:
from pyspark.sql.functions import col
from datetime import datetime

# Получаем значения из предыдущих тасков
hourly_clean_count = dbutils.jobs.taskValues.get(
    taskKey="02_clean_hourly",
    key="hourly_clean_count",
    debugValue=0
)

daily_clean_count = dbutils.jobs.taskValues.get(
    taskKey="03_clean_daily",
    key="daily_clean_count",
    debugValue=0
)

# Читаем silver таблицы для дополнительных проверок

hourly_df = spark.table("weather_project.silver.hourly_clean")
daily_df = spark.table("weather_project.silver.daily_clean")

# Проверяем количество записей в таблицах

checks = [
    {
        "check_name": "hourly_rows",
        "condition": hourly_clean_count > 0,
        "expected": "> 0",
        "actual": str(hourly_clean_count)
    },
    {
        "check_name": "daily_rows",
        "condition": daily_clean_count > 0,
        "expected": "> 0",
        "actual": str(daily_clean_count)
    },
    {
        "check_name": "temperature_range",
        "condition": hourly_df.filter(
            (col("temperature") <= -80) | (col("temperature") >= 70)
        ).count() == 0,
        "expected": "-80 < temp < 70",
        "actual": str(hourly_df.filter(
            (col("temperature") <= -80) | (col("temperature") >= 70)
        ).count()) + " invalid rows"
    },
    {
        "check_name": "date_nulls",
        "condition": hourly_df.filter(col("event_date").isNull()).count() == 0,
        "expected": "none",
        "actual": str(hourly_df.filter(col("event_date").isNull()).count()) + " nulls"
    }
]

# Сохраняем результаты проверок и печатаем по каждому значению
results = []
for check in checks:
    status = "PASS" if check["condition"] else "FAIL"
    print(f"{'GOOD' if status == 'PASS' else 'BAD'} {check['check_name']}: {status} (expected: {check['expected']}, actual: {check['actual']})")
    results.append((
        check["check_name"],
        status,
        check["expected"],
        check["actual"],
        datetime.now()
    ))

# Сохраняем в monitoring таблицу
schema = ["check_name", "status", "expected", "actual", "checked_at"]
monitoring_df = spark.createDataFrame(results, schema)

monitoring_df.write \
    .mode("append") \
    .saveAsTable("weather_project.monitoring.quality_checks")

display(monitoring_df)